# Anvil clouds fraction

In [1]:
from getpass import getuser # Libaray to copy things
from tempfile import NamedTemporaryFile, TemporaryDirectory 

# calculation
#import metpy.calc as mpcalc

# scipy
from scipy import stats
from scipy.ndimage import measurements
from scipy import ndimage
from scipy.optimize import curve_fit

# for plot
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.feature as cfeature

import easygems.healpix as egh
import healpy as hp

# basic
from pathlib import Path # Object oriented libary to deal with paths
import netCDF4 as nc
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import glob

In [2]:
def get_amazon_index(nside=12**2, nest=True):
    amazon = xr.open_dataset(
        "/work/mh0731/m300948/AMDEF/REGRID_BC/masking_files/AMAZON_Biome.nc"
    ).stack(cell=("latitude", "longitude"))
    
    is_amazon = amazon.where(amazon.AMAZON_BIOMES == 0, drop=True)
    
    return np.unique(hp.ang2pix(nside, is_amazon.longitude.values, is_amazon.latitude.values, nest=nest, lonlat=True)).astype(int)


In [3]:
def samerica(ds):
     return (ds.lat >= -60) & (ds.lat <= 20) & (ds.lon >= 260) & (ds.lon <= 340)
def amazon(ds):
    dset_bd = xr.open_dataset('/work/mh0731/m300948/AMDEF/REGRID_BC/masking_files/AMAZON_Biome.nc')
    AMZ_BD = dset_bd.AMAZON_BIOMES
    AMZ_BD['longitude'] = (AMZ_BD['longitude'] + 360) % 360 # change from -180-180 to 0-360
    grid_amz = AMZ_BD.interp(latitude=ds.lat, longitude=ds.lon)
    return (ds.lat >= -30) & (ds.lat <= 20) & (ds.lon >= 270) & (ds.lon <= 330) & (grid_amz==0)

In [4]:
def colormap_create(cmap, cnumber, loc_boundary_low, loc_boundary_high):
    get_cmap = cm.get_cmap(cmap,cnumber) 
    cmap_edit = get_cmap(np.linspace(0,1,cnumber))
    white = np.array([255/256, 255/256, 255/256, 1])
    cmap_edit[loc_boundary_low:loc_boundary_high] = white
    cmap_new = matplotlib.colors.ListedColormap(cmap_edit)
    return cmap_new

In [5]:
BrBG_new = colormap_create('BrBG',21,10,11)
YlOrRd_new = colormap_create('YlOrRd',21,0,1)

/tmp/ipykernel_3178181/511728102.py:2: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  get_cmap = cm.get_cmap(cmap,cnumber)


# Load datasets

In [6]:
# ICON CTL (5km)
import intake
cat = intake.open_catalog("https://data.nextgems-h2020.eu/catalog.yaml")
data_ctl  = cat['ICON.C5.AMIP_CNTL'](time="P1D",zoom=8, grid=f"R02B08", chunks="auto").to_dask().pipe(egh.attach_coords) # 1979-1997 (19 years) original: zoom8

/home/m/m300948/.conda/envs/easy/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


In [7]:
dx_p4k  = cat['ICON.C5.AMIP_P4K'](time="P1D", zoom=8, grid=f"R02B08", chunks="auto").to_dask().pipe(egh.attach_coords) # 1979-1993 (15 years)

/home/m/m300948/.conda/envs/easy/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


In [8]:
dx_4co2 = cat['ICON.C5.AMIP_4CO2'](time="P1D",zoom=8, grid=f"R02B08", chunks="auto").to_dask().pipe(egh.attach_coords) # 1980-1994 (15 years)

/home/m/m300948/.conda/envs/easy/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


In [9]:
# ICON-amip
dx_ctl = data_ctl.sel(time=slice("1980", "1994")) 
dx_p4k = dx_p4k.sel(time=slice("1979", "1993"))
dx_4co2 = dx_4co2.sel(time=slice("1980", "1994"))

In [17]:
dx_ctl

<xarray.Dataset> Size: 6TB
Dimensions:             (time: 5479, cell: 786432, level_full: 26,
                         level_half: 26)
Coordinates:
  * time                (time) datetime64[ns] 44kB 1980-01-01 ... 1994-12-31
  * cell                (cell) int64 6MB 0 1 2 3 ... 786428 786429 786430 786431
    lat                 (cell) float64 6MB 0.1492 0.2984 ... -0.2984 -0.1492
    lon                 (cell) float64 6MB 45.0 45.18 44.82 ... 314.8 315.0
  * level_full          (level_full) float64 208B 14.0 21.0 25.0 ... 89.0 90.0
  * level_half          (level_half) float64 208B 14.0 21.0 25.0 ... 89.0 90.0
    healpix             int64 8B 0
Data variables: (12/47)
    clivi               (time, cell) float32 17GB dask.array<chunksize=(366, 81920), meta=np.ndarray>
    cllvi               (time, cell) float32 17GB dask.array<chunksize=(366, 81920), meta=np.ndarray>
    hus2m               (time, cell) float32 17GB dask.array<chunksize=(366, 81920), meta=np.ndarray>
    hfls                (time, cell) float32 17GB dask.array<chunksize=(366, 81920), meta=np.ndarray>
    hfss                (time, cell) float32 17GB dask.array<chunksize=(366, 81920), meta=np.ndarray>
    pr                  (time, cell) float32 17GB dask.array<chunksize=(366, 81920), meta=np.ndarray>
    ...                  ...
    phalf               (time, level_half, cell) float32 448GB dask.array<chunksize=(64, 8, 32768), meta=np.ndarray>
    cell_elevation      (cell) float64 6MB dask.array<chunksize=(786432,), meta=np.ndarray>
    cell_sea_land_mask  (cell) int32 3MB dask.array<chunksize=(786432,), meta=np.ndarray>
    zg                  (level_full, cell) float32 82MB dask.array<chunksize=(26, 786432), meta=np.ndarray>
    zghalf              (level_half, cell) float32 82MB dask.array<chunksize=(26, 786432), meta=np.ndarray>
    dzghalf             (level_full, cell) float32 82MB dask.array<chunksize=(26, 786432), meta=np.ndarray>
Attributes:
    CDI:                       Climate Data Interface version 2.4.0 (https://...
    Conventions:               CF-1.6
    source:                    https://gitlab.dkrz.de/icon/icon-mpim.git@6684...
    institution:               Max Planck Institute for Meteorology/Deutscher...
    title:                     ICON simulation
    references:                see MPIM/DWD publications
    comment:                   Lukas Kluft (kluftluka) on nid006406 (Linux 5....
    cdo_bitrounding_numbits:   13
    CDO:                       Climate Data Operators version 2.4.0 (https://...
    cdo_openmp_thread_number:  4
    history:                   Wed Mar 13 20:19:59 2024: ncrename -d cells,ce...
    NCO:                       netCDF Operators version 5.0.1 (Homepage = htt...

### ICON NGC

In [10]:
# NGC4
dx_ngc = cat.ICON.ngc4008(zoom=8).to_dask().pipe(egh.attach_coords)

/home/m/m300948/.conda/envs/easy/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


In [11]:
dx_ngc_start = dx_ngc.sel(time=slice('2024-01-01', '2028-12-31')) # 5 years
dx_ngc_end = dx_ngc.sel(time=slice('2045-01-01', '2049-12-31')) # 5years

## Amazon

In [12]:
is_amazon = get_amazon_index(nside=egh.get_nside(data_ctl))
dx_ctl_amz = data_ctl.isel(cell=is_amazon).sel(time=slice("1980", "1994"))  # 1979-1997
dx_p4k_amz = dx_p4k.isel(cell=is_amazon).sel(time=slice("1979", "1993")) # 1979-1993
dx_4co2_amz = dx_4co2.isel(cell=is_amazon).sel(time=slice("1980", "1994")) # 1980-1994

In [13]:
# NGC4
is_amazon_ngc = get_amazon_index(nside=egh.get_nside(dx_ngc))
dx_ngc_start_amz = dx_ngc_start.isel(cell=is_amazon_ngc)
dx_ngc_end_amz = dx_ngc_end.isel(cell=is_amazon_ngc)

# Atmospheric driness

In [18]:
prw_ctl_amz = dx_ctl_amz.prw
prw_4co2_amz = dx_4co2_amz.prw
prw_p4k_amz = dx_p4k_amz.prw

In [19]:
print(prw_ctl_amz.mean(dim=('time','cell')).values)

array(37.749207, dtype=float32)

In [20]:
print(prw_4co2_amz.mean(dim=('time','cell')).values)

38.572136


In [21]:
print(prw_p4k_amz.mean(dim=('time','cell')).values)

52.200035


In [26]:
prw_p4k_amz

<xarray.DataArray 'prw' (time: 5478, cell: 10611)> Size: 233MB
dask.array<getitem, shape=(5478, 10611), dtype=float32, chunksize=(366, 10611), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 44kB 1979-01-02 1979-01-03 ... 1993-12-31
  * cell     (cell) int64 85kB 475205 475207 475213 ... 786402 786408 786410
    lat      (cell) float64 85kB -17.58 -17.43 -17.27 ... -1.492 -1.343 -1.194
    lon      (cell) float64 85kB 294.4 294.3 294.1 293.9 ... 314.1 313.9 313.8
    healpix  int64 8B 0
Attributes:
    standard_name:  atmosphere_mass_content_of_water_vapor
    long_name:      water vapor path
    units:          kg m-2
    param:          64.1.0
    grid_mapping:   healpix

# Clouds types

In [15]:
dx_ctl_amz.clw

<xarray.DataArray 'clw' (time: 5479, level_full: 26, cell: 10611)> Size: 6GB
dask.array<getitem, shape=(5479, 26, 10611), dtype=float32, chunksize=(64, 10, 10611), chunktype=numpy.ndarray>
Coordinates:
  * time        (time) datetime64[ns] 44kB 1980-01-01 1980-01-02 ... 1994-12-31
  * level_full  (level_full) float64 208B 14.0 21.0 25.0 29.0 ... 87.0 89.0 90.0
  * cell        (cell) int64 85kB 475205 475207 475213 ... 786402 786408 786410
    lat         (cell) float64 85kB -17.58 -17.43 -17.27 ... -1.343 -1.194
    lon         (cell) float64 85kB 294.4 294.3 294.1 ... 314.1 313.9 313.8
    healpix     int64 8B 0
Attributes:
    standard_name:  clw
    long_name:      specific cloud water content
    units:          kg kg-1
    param:          22.1.0
    grid_mapping:   healpix

In [ ]:
# Cloud ice
print(dx_ctl_amz.cli.mean(dim=('time','cell')).sum('level_full').values)
print(dx_4co2_amz.cli.mean(dim=('time','cell')).sum('level_full').values)
print(dx_p4k_amz.cli.mean(dim=('time','cell')).sum('level_full').values)

2.5832607e-05
2.9732497e-05
2.008509e-05


In [28]:
# Cloud ice
print(dx_ctl_amz.clw.mean(dim=('time','cell')).sum('level_full').values)
print(dx_4co2_amz.clw.mean(dim=('time','cell')).sum('level_full').values)
print(dx_p4k_amz.clw.mean(dim=('time','cell')).sum('level_full').values)

0.0002484333
0.00024515414
0.00023536774


## OLR

In [38]:
dx_ctl_amz.rlut

<xarray.DataArray 'rlut' (time: 5479, cell: 10611)> Size: 233MB
dask.array<getitem, shape=(5479, 10611), dtype=float32, chunksize=(366, 10611), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 44kB 1980-01-01 1980-01-02 ... 1994-12-31
  * cell     (cell) int64 85kB 475205 475207 475213 ... 786402 786408 786410
    lat      (cell) float64 85kB -17.58 -17.43 -17.27 ... -1.492 -1.343 -1.194
    lon      (cell) float64 85kB 294.4 294.3 294.1 293.9 ... 314.1 313.9 313.8
    healpix  int64 8B 0
Attributes:
    standard_name:  toa_outgoing_longwave_flux
    long_name:      toa outgoing longwave radiation
    units:          W m-2
    param:          4.5.0
    grid_mapping:   healpix

In [36]:
# Deep convection OLR threshold < 240 W/m2, how to calculate the sum of area?
print(dx_ctl_amz.rlut.where(dx_ctl_amz.rlut < 240).mean(dim=("time","cell")).values)
print(dx_4co2_amz.rlut.where(dx_4co2_amz.rlut < 240).mean(dim=("time","cell")).values)
print(dx_p4k_amz.rlut.where(dx_p4k_amz.rlut < 240).mean(dim=("time","cell")).values)

212.0655
207.76007
218.08012


In [37]:
# Deep convection OLR threshold >= 240 W/m2, how to calculate the sum of area?
print(dx_ctl_amz.rlut.where(dx_ctl_amz.rlut >= 240).mean(dim=("time","cell")).values)
print(dx_4co2_amz.rlut.where(dx_4co2_amz.rlut >= 240).mean(dim=("time","cell")).values)
print(dx_p4k_amz.rlut.where(dx_p4k_amz.rlut >= 240).mean(dim=("time","cell")).values)

267.47827
263.67215
274.31848


In [39]:
import numpy as np
import xarray as xr

EARTH_R = 6_371_000.0  # m

def healpix_pixel_area(npix_total, R=EARTH_R):
    """Equal area per HEALPix pixel for the FULL grid."""
    return 4.0 * np.pi * R**2 / float(npix_total)  # m^2 npix_total = 786,432

def olr_area_stats_amazon(
    rlut_amz: xr.DataArray,
    thr=240.0,
    npix_total=None,          # <-- REQUIRED unless you have cell_area
    cell_area=None,           # optional xr.DataArray (cell,) in m^2
    time_dim="time",
    cell_dim="cell",
):
    """
    rlut_amz: Amazon-subset rlut (time, cell)
    Returns:
      area_below (time): total Amazon area with rlut<thr [m^2]
      frac_below (time): fraction of Amazon area with rlut<thr [-]
      freq_map (cell): frequency (0..1) each cell is below thr
      mask (time, cell): boolean
    """
    if cell_area is None:
        if npix_total is None:
            raise ValueError(
                "You are using an Amazon subset. Provide `npix_total` from the ORIGINAL full HEALPix grid "
                "(or pass `cell_area` as a (cell,) DataArray in m^2)."
            )
        a = healpix_pixel_area(npix_total) # array with the size of Amazon area with the value of single grid's area
        cell_area = xr.DataArray(
            np.full((rlut_amz.sizes[cell_dim],), a, dtype=np.float64), 
            dims=(cell_dim,),
            coords={cell_dim: rlut_amz[cell_dim] if cell_dim in rlut_amz.coords else np.arange(rlut_amz.sizes[cell_dim])},
            name="cell_area",
            attrs={"units": "m2"},
        )

    # boolean mask where rlut is below threshold
    mask = rlut_amz > thr

    # Convective area only, time series (m^2)
    area_below = (mask * cell_area).sum(dim=cell_dim)

    # total Amazon area (m^2)
    amazon_area = cell_area.sum(dim=cell_dim)

    # Convective clouds fraction, time series
    frac_below = area_below / amazon_area

    # frequency per pixel (0..1)
    freq_map = mask.mean(dim=time_dim)

    return area_below, frac_below, freq_map, mask


In [36]:
a = healpix_pixel_area(786432) # how to calculate the area
a

648580515.4289097

In [40]:
rlut_amz = dx_ctl_amz["rlut"] if isinstance(dx_ctl_amz, xr.Dataset) else dx_ctl_amz

# IMPORTANT: set this to the full-grid number of HEALPix cells (before Amazon selection)
NPIX_TOTAL = 786432  # <-- example if nside=1024
# If your nside is different, change it accordingly (nside=512 -> 12*512**2, etc.)

area_below, frac_below, freq_map, mask = olr_area_stats_amazon(
    rlut_amz,
    thr=240.0,
    npix_total=NPIX_TOTAL,
)

print("Mean Amazon area with rlut<240 (km^2):", float(area_below.mean("time") / 1e6))
print("Mean fraction of Amazon with rlut<240:", float(frac_below.mean("time")))


Mean Amazon area with rlut<240 (km^2): 3631892.499701831
Mean fraction of Amazon with rlut<240: 0.5277312029830435


In [41]:
rlut_amz = dx_4co2_amz["rlut"] if isinstance(dx_4co2_amz, xr.Dataset) else dx_4co2_amz

# IMPORTANT: set this to the full-grid number of HEALPix cells (before Amazon selection)
NPIX_TOTAL = 786432   # <-- example if nside=1024
# If your nside is different, change it accordingly (nside=512 -> 12*512**2, etc.)

area_below, frac_below, freq_map, mask = olr_area_stats_amazon(
    rlut_amz,
    thr=240.0,
    npix_total=NPIX_TOTAL,
)

print("Mean Amazon area with rlut<240 (km^2):", float(area_below.mean("time") / 1e6))
print("Mean fraction of Amazon with rlut<240:", float(frac_below.mean("time")))


Mean Amazon area with rlut<240 (km^2): 2828192.453807865
Mean fraction of Amazon with rlut<240: 0.41094977509332203


In [43]:
rlut_amz = dx_p4k_amz["rlut"] if isinstance(dx_p4k_amz, xr.Dataset) else dx_p4k_amz

# IMPORTANT: set this to the full-grid number of HEALPix cells (before Amazon selection)
NPIX_TOTAL = 786432   # <-- example if nside=1024
# If your nside is different, change it accordingly (nside=512 -> 12*512**2, etc.)

area_below, frac_below, freq_map, mask = olr_area_stats_amazon(
    rlut_amz,
    thr=240.0,
    npix_total=NPIX_TOTAL,
)

print("Mean Amazon area with rlut<240 (km^2):", float(area_below.mean("time") / 1e6))
print("Mean fraction of Amazon with rlut<240:", float(frac_below.mean("time")))

Mean Amazon area with rlut<240 (km^2): 4791853.977593455
Mean fraction of Amazon with rlut<240: 0.6962791063672963
